# Customer Churn Prediction

A simple machine learning project to predict whether a telecom customer will churn.

## Import libraries

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, ConfusionMatrixDisplay, classification_report

## Load dataset

In [ ]:
df = pd.read_csv('../data/customer_churn.csv')
df.head()

## Basic dataset checks

In [ ]:
print('Shape:', df.shape)
print('\nColumns:', df.columns.tolist())
df.info()

In [ ]:
df.describe(include='all')

In [ ]:
print('Missing values:')
print(df.isnull().sum())
print('\nDuplicate rows:', df.duplicated().sum())

## Data preparation

In [ ]:
data = df.drop(columns='CustomerID').copy()
data['Churn'] = data['Churn'].map({'No': 0, 'Yes': 1})
data.head()

## Exploratory Data Analysis

In [ ]:
df['Churn'].value_counts().plot(kind='bar', figsize=(6, 4))
plt.title('Customer Churn')
plt.xlabel('Churn')
plt.ylabel('Customers')
plt.xticks(rotation=0)
plt.show()

In [ ]:
pd.crosstab(df['Contract'], df['Churn']).plot(kind='bar', figsize=(8, 4))
plt.title('Churn by Contract Type')
plt.xlabel('Contract')
plt.ylabel('Customers')
plt.xticks(rotation=0)
plt.show()

In [ ]:
for churn in ['No', 'Yes']:
    plt.hist(df[df['Churn'] == churn]['Tenure'], bins=8, alpha=0.6, label=churn)

plt.title('Tenure by Churn')
plt.xlabel('Tenure')
plt.ylabel('Customers')
plt.legend()
plt.show()

In [ ]:
df.groupby('Churn')['SupportCalls'].mean().plot(kind='bar', figsize=(6, 4))
plt.title('Average Support Calls by Churn')
plt.xlabel('Churn')
plt.ylabel('Average Support Calls')
plt.xticks(rotation=0)
plt.show()

## Encoding and train-test split

In [ ]:
X = data.drop(columns='Churn')
y = data['Churn']

X = pd.get_dummies(X, drop_first=True, dtype=int)
X.head()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print('Training data:', X_train.shape)
print('Testing data:', X_test.shape)

## Scaling

In [ ]:
scale_cols = ['Tenure', 'MonthlyCharges', 'TotalCharges', 'SupportCalls']

X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

scaler = StandardScaler()
X_train_scaled[scale_cols] = scaler.fit_transform(X_train[scale_cols])
X_test_scaled[scale_cols] = scaler.transform(X_test[scale_cols])

## Logistic Regression

In [ ]:
log_model = LogisticRegression(max_iter=1000)
log_model.fit(X_train_scaled, y_train)

log_pred = log_model.predict(X_test_scaled)

In [ ]:
print('Accuracy:', round(accuracy_score(y_test, log_pred), 3))
print('Precision:', round(precision_score(y_test, log_pred, zero_division=0), 3))
print('Recall:', round(recall_score(y_test, log_pred, zero_division=0), 3))
print('F1 Score:', round(f1_score(y_test, log_pred, zero_division=0), 3))
print('\n', classification_report(y_test, log_pred, zero_division=0))

In [ ]:
cm = confusion_matrix(y_test, log_pred)
ConfusionMatrixDisplay(cm, display_labels=['No Churn', 'Churn']).plot()
plt.title('Logistic Regression')
plt.show()

## Decision Tree

In [ ]:
tree_model = DecisionTreeClassifier(max_depth=3, random_state=42)
tree_model.fit(X_train, y_train)

tree_pred = tree_model.predict(X_test)

In [ ]:
results = pd.DataFrame({
    'Model': ['Logistic Regression', 'Decision Tree'],
    'Accuracy': [accuracy_score(y_test, log_pred), accuracy_score(y_test, tree_pred)],
    'Precision': [precision_score(y_test, log_pred, zero_division=0), precision_score(y_test, tree_pred, zero_division=0)],
    'Recall': [recall_score(y_test, log_pred, zero_division=0), recall_score(y_test, tree_pred, zero_division=0)],
    'F1 Score': [f1_score(y_test, log_pred, zero_division=0), f1_score(y_test, tree_pred, zero_division=0)]
})

results.round(3)

## Test with a new customer

In [ ]:
new_customer = pd.DataFrame([{
    'Gender': 'Male',
    'SeniorCitizen': 'No',
    'Partner': 'No',
    'Dependents': 'No',
    'Tenure': 6,
    'InternetService': 'Fiber optic',
    'Contract': 'Month-to-month',
    'PaymentMethod': 'Electronic check',
    'MonthlyCharges': 95.0,
    'TotalCharges': 570.0,
    'SupportCalls': 5
}])

new_customer = pd.get_dummies(new_customer, dtype=int)
new_customer = new_customer.reindex(columns=X.columns, fill_value=0)

new_customer_scaled = new_customer.copy()
new_customer_scaled[scale_cols] = scaler.transform(new_customer[scale_cols])

prediction = log_model.predict(new_customer_scaled)[0]
probability = log_model.predict_proba(new_customer_scaled)[0][1]

print('Prediction:', 'Churn' if prediction == 1 else 'No Churn')
print('Churn probability:', round(probability * 100, 2), '%')

## Conclusion

This project compares Logistic Regression and Decision Tree for customer churn prediction. The included dataset is small, so the project is mainly for learning the machine learning workflow.